# Phase 4 — SHAP Explainability
**Tool:** SHAP (SHapley Additive exPlanations) with `TreeExplainer` for XGBoost.  
**Goal:** Quantify each feature's contribution to the model's predictions at both global and per-class levels, and explain individual predictions.

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib, os, warnings
warnings.filterwarnings('ignore')

DATA_PATH   = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\data"
MODELS_PATH = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\models"
PLOTS_PATH  = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\reports"
os.makedirs(PLOTS_PATH, exist_ok=True)

## 2. Load Data & Model

In [ ]:
df  = pd.read_parquet(os.path.join(DATA_PATH, 'clean_data.parquet'))
le  = joblib.load(os.path.join(DATA_PATH, 'label_encoder.pkl'))
xgb = joblib.load(os.path.join(MODELS_PATH, 'xgb_model.pkl'))

X = df.drop('Label', axis=1)
y = df['Label']
feature_names = X.columns.tolist()
print(f"✅ {len(X):,} rows  ·  {len(feature_names)} features")

## 3. Sample & Compute SHAP Values

In [ ]:
sample_idx = np.random.choice(len(X), 5000, replace=False)
X_sample   = X.iloc[sample_idx]
y_sample   = y.iloc[sample_idx]

explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_sample)
print(f"✅ SHAP values shape: {np.array(shap_values).shape}")  # (n_classes, n_samples, n_features)

## 4. Global Feature Importance

In [ ]:
shap_importance = np.mean(np.abs(shap_values), axis=2)
mean_importance = np.mean(shap_importance, axis=0)

importance_df = pd.DataFrame({
    'feature':    feature_names,
    'importance': mean_importance
}).sort_values('importance', ascending=False)

print("Top 15 features:")
print(importance_df.head(15).to_string(index=False))

top20  = importance_df.head(20)
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 20))

plt.figure(figsize=(12, 8))
plt.barh(range(20), top20['importance'].values, color=colors)
plt.yticks(range(20), top20['feature'].values, fontsize=10)
plt.xlabel('Mean |SHAP Value|', fontsize=12)
plt.title('Top 20 Most Important Features (Global)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis(); plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_PATH, 'shap_global_importance.png'), dpi=150)
plt.show()

## 5. Per-Class Feature Importance

In [ ]:
interesting_classes = {
    2: 'DDoS', 4: 'DoS Hulk', 7: 'FTP-Patator',
    10: 'PortScan', 12: 'Web Attack - Brute Force'
}

fig, axes = plt.subplots(1, len(interesting_classes), figsize=(20, 6))

for idx, (class_idx, class_name) in enumerate(interesting_classes.items()):
    class_shap = shap_values[:, :, class_idx]
    mean_abs   = np.mean(np.abs(class_shap), axis=0)
    top10_idx  = np.argsort(mean_abs)[-10:][::-1]
    axes[idx].barh(range(10), mean_abs[top10_idx],
                   color=plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 10)))
    axes[idx].set_yticks(range(10))
    axes[idx].set_yticklabels([feature_names[i] for i in top10_idx], fontsize=7)
    axes[idx].set_title(class_name, fontsize=10, fontweight='bold')
    axes[idx].invert_yaxis(); axes[idx].grid(axis='x', alpha=0.3)

plt.suptitle('Top 10 Features per Attack Type (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_PATH, 'shap_per_class.png'), dpi=150)
plt.show()

## 6. Single Prediction Explanation

In [ ]:
ddos_label  = le.transform(['DDoS'])[0]
ddos_mask   = y_sample.values == ddos_label
ddos_samples = X_sample[ddos_mask]

if len(ddos_samples) > 0:
    first_idx   = np.where(ddos_mask)[0][0]
    single_shap = shap_values[first_idx, :, ddos_label]
    top_idx     = np.argsort(np.abs(single_shap))[-10:][::-1]
    top_features = [feature_names[i] for i in top_idx]
    top_vals     = single_shap[top_idx]

    colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in top_vals]
    plt.figure(figsize=(10, 6))
    plt.barh(range(10), top_vals, color=colors, edgecolor='black', lw=0.5)
    plt.yticks(range(10), top_features, fontsize=10)
    plt.axvline(0, color='black', lw=1)
    plt.xlabel('SHAP Value  (red → attack  |  green → normal)', fontsize=10)
    plt.title('Single Prediction Explanation — DDoS', fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis(); plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_PATH, 'shap_single_prediction.png'), dpi=150)
    plt.show()
else:
    print("No DDoS samples in this random sample — re-run or increase sample size")

## 7. Save SHAP Artefacts

In [ ]:
joblib.dump(shap_values,   os.path.join(MODELS_PATH, 'shap_values.pkl'))
joblib.dump(importance_df, os.path.join(MODELS_PATH, 'shap_importance.pkl'))
print("✅ shap_values.pkl  saved")
print("✅ shap_importance.pkl  saved")